In [7]:
import base64
import requests
from PIL import Image
import os

USDA_API_KEY = os.getenv("USDA_API_KEY") or "UibX72HfLzbSnDqkyBICmaL1RoeERHxgoXlBWadW"

def get_base64_image(image):
    response = requests.get(image)
    if response.status_code == 200:
        encoded_image = base64.b64encode(response.content).decode("utf-8")
    return encoded_image

def identify_food_items_llava(image):
    img_b64 = get_base64_image(image)
    response = requests.post("http://localhost:11434/api/generate", json={
        "model": "llava",
        "prompt": "List all recognizable food items in this image. Just list them, comma-separated.",
        "images": [img_b64],
        "stream": False
    })
    return response.json()["response"]

def get_calories_usda(food_name):
    search_url = f"https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "query": food_name,
        "pageSize": 1,
        "api_key": USDA_API_KEY
    }
    response = requests.get(search_url, params=params)
    data = response.json()
    try:
        item = data['foods'][0]
        calories = next((n["value"] for n in item["foodNutrients"] if n["nutrientName"] == "Energy"), None)
        return calories or 0
    except (KeyError, IndexError):
        return 0

def estimate_total_calories(image):
    caption = identify_food_items_llava(image)
    food_items = [item.strip().lower() for item in caption.split(",")]
    results = []

    for item in food_items:
        cals = get_calories_usda(item)
        results.append({
            "name": item,
            "estimated_calories_per_100g": cals
        })

    total = sum(r["estimated_calories_per_100g"] for r in results)

    return {
        "caption": caption,
        "foods": results,
        "estimated_total_calories": total
    }

# 🧪 Example usage
if __name__ == "__main__":
    # image = 'https://www.baltana.com/files/wallpapers-2/Food-HD-Pictures-04863.jpg'
    image = 'https://thumbs.dreamstime.com/b/south-indian-food-platter-idli-sambhar-vada-dosa-chutneys-144914344.jpg'
    result = estimate_total_calories(image)
    print(result)


{'caption': ' Naan, chutney, pakora, sambar, vada, chips, poppadum, dosa, curd, chutney powder, idli, tomato, onion, coriander leaves, dhokla ', 'foods': [{'name': 'naan', 'estimated_calories_per_100g': 288}, {'name': 'chutney', 'estimated_calories_per_100g': 246}, {'name': 'pakora', 'estimated_calories_per_100g': 125}, {'name': 'sambar', 'estimated_calories_per_100g': 86}, {'name': 'vada', 'estimated_calories_per_100g': 266}, {'name': 'chips', 'estimated_calories_per_100g': 494}, {'name': 'poppadum', 'estimated_calories_per_100g': 464}, {'name': 'dosa', 'estimated_calories_per_100g': 184}, {'name': 'curd', 'estimated_calories_per_100g': 61}, {'name': 'chutney powder', 'estimated_calories_per_100g': 246}, {'name': 'idli', 'estimated_calories_per_100g': 128}, {'name': 'tomato', 'estimated_calories_per_100g': 0}, {'name': 'onion', 'estimated_calories_per_100g': 289}, {'name': 'coriander leaves', 'estimated_calories_per_100g': 95.0}, {'name': 'dhokla', 'estimated_calories_per_100g': 360}]